# 散射光路 + DNN + GAN 仿真系统说明

当前项目实现了一套从散射光路仿真到神经网络重建、再到生成对抗细化的实验系统。

当前系统完成了一条从光路污染到神经网络重建，再到 GAN 对抗细化的实验链路：

```text
clean image
  -> coherent field encoding
  -> phase screen / particle corruption
  -> free-space propagation
  -> D2NN intensity readout
  -> U-Net reconstruction
  -> PatchGAN refinement
  -> metrics and figures
```

下面按流程说明各个模块的作用、使用的方法，以及对应的主要代码。


## 1. 当前已经完成的内容

当前项目已经完成以下内容：

1. 构建了 coherent optical forward model。
2. 实现了相位屏和粒子遮挡两类污染方式。
3. 实现了自由空间传播和单层 D2NN intensity readout。
4. 构建了 paired dataset：`clean` 与 `d2nn_intensity` 成对出现。
5. 训练了 U-Net reconstructor，从污染强度图恢复 clean image。
6. 在 U-Net 之后加入 PatchGAN adversarial refinement。
7. 在 GPU 上完成了 U-Net 与 U-Net+GAN 的初步比较。

这套系统目前作为 prototype experimental pipeline 使用，输出包括 metrics、figures、checkpoints、manifest 和 history。


## 2. 主要代码文件

| 文件 | 作用 |
|---|---|
| `experiment.py` | 统一实验入口，包含 `d2nn`、`unet`、`gan`、`compare`、`full` 子命令。 |
| `d2nn.py` | 复光场编码、相位屏、粒子遮挡、角谱传播、单层 D2NN。 |
| `coherent_data.py` | 生成成对样本：clean、dirty intensity、dirty phase、D2NN intensity。 |
| `unet.py` | U-Net reconstructor。 |
| `patchgan.py` | 条件 PatchGAN discriminator。 |
| `losses.py` | L1、negative Pearson、SSIM-like、Fourier consistency 等 loss。 |
| `metrics.py` | L1、MSE、PSNR、SSIM、Pearson。 |
| `runtime.py` | seed、device、JSON、输出目录管理。 |


## 3. 光场编码

输入图像是灰度强度图 `x`。仿真中把它转成复光场：

```text
amplitude = sqrt(x)
phase = 0
field = amplitude * exp(i * phase)
```

对应核心代码在 `d2nn.py`：

```python
field = image_to_complex_field(clean)
intensity = field_intensity(field)
```

这样处理后，之后可以在复数光场上施加相位扰动和传播。


In [ ]:
from pathlib import Path
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
from PIL import Image

from coherent_data import simulate_coherent_observation
from data import build_torchvision_dataset

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "experiment.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

dataset = build_torchvision_dataset(
    name="MNIST",
    root=PROJECT_ROOT / "data",
    train=False,
    image_size=64,
    download=False,
)
clean, label = dataset[0]
print("clean shape:", tuple(clean.shape))
print("label:", int(label))


## 4. 杂信号生成方法

当前项目用了两类污染方式。

### 4.1 随机相位屏

随机相位屏改变光场相位：

```python
phase_screen = make_random_phase_screen(config.field_shape, seed=seed)
scattered_field = apply_phase_screen(field, phase_screen)
```

相位扰动经过自由空间传播后，会在强度图上形成 speckle-like pattern。

### 4.2 粒子遮挡

粒子遮挡改变局部振幅：

```python
particle_mask = make_amplitude_particles(config.field_shape, seed=seed)
scattered_field = apply_amplitude_particles(field, particle_mask)
```

这类污染对应局部吸收或遮挡。


## 5. 自由空间传播与 D2NN intensity readout

污染后的复光场先经过角谱传播：

```python
dirty_field = AngularSpectrumPropagator(config).propagate(scattered_field)
```

然后进入一个固定的 phase-only D2NN layer：

```python
output_field = SingleLayerD2NN(config, seed=d2nn_seed)(dirty_field)
d2nn_intensity = field_intensity(output_field)
```

`d2nn_intensity` 是 U-Net 和 GAN 的主要输入。


In [ ]:
sample = simulate_coherent_observation(clean, corruption="phase", seed=42, d2nn_seed=7961)
for key, value in sample.items():
    print(key, tuple(value.shape), value.dtype)


下面这组图展示同一个样本在光路中的几个状态。


In [ ]:
panels = [
    ("clean", sample["clean"]),
    ("dirty intensity", sample["dirty_intensity"]),
    ("dirty phase", sample["dirty_phase"]),
    ("D2NN intensity", sample["d2nn_intensity"]),
]

fig, axes = plt.subplots(1, len(panels), figsize=(10, 2.6))
for axis, (title, image) in zip(axes, panels):
    axis.imshow(image[0].detach().cpu(), cmap="gray", vmin=0, vmax=1)
    axis.set_title(title)
    axis.set_axis_off()
fig.tight_layout()


## 6. DNN 重建方法

DNN 阶段使用 U-Net。输入是 `d2nn_intensity`，监督目标是 `clean`：

```text
source = batch["d2nn_intensity"]
target = batch["clean"]
prediction = UNet(source)
```

基础训练使用 L1 reconstruction loss：

```python
loss = L1(prediction, target)
loss.backward()
optimizer.step()
```

GPU 训练结果显示 U-Net 已经学到反演关系。


In [ ]:
run_dir = PROJECT_ROOT / "outputs" / "gpu_phase_comparison"
unet_history = json.loads((run_dir / "unet" / "history.json").read_text(encoding="utf-8"))
first = unet_history[0]
last = unet_history[-1]

print("U-Net train L1:", first["train"]["l1"], "->", last["train"]["l1"])
print("U-Net eval SSIM:", first["eval"]["ssim"], "->", last["eval"]["ssim"])
print("U-Net eval Pearson:", first["eval"]["pearson"], "->", last["eval"]["pearson"])


## 7. GAN 生成对抗部分

GAN 阶段使用 U-Net 作为 generator，PatchGAN 作为 discriminator。

Generator：

```text
G(y) -> reconstructed image
```

Discriminator：

```text
D(y, real clean image) -> real/fake logits
D(y, generated image)  -> real/fake logits
```

Generator loss 由两部分组成：

```text
generator_loss = reconstruction_loss + adversarial_weight * adversarial_loss
```

当前实验中 `adversarial_weight = 0.01`。GAN generator 从已经训练好的 U-Net checkpoint 初始化。


## 8. U-Net 与 U-Net+GAN 结果

下面读取已经完成的 GPU 实验结果。


In [ ]:
comparison = json.loads((run_dir / "comparison" / "comparison.json").read_text(encoding="utf-8"))
metric_order = ["l1", "mse", "psnr", "ssim", "pearson"]

print(f"{'metric':<8} {'U-Net':>10} {'GAN':>10} {'delta':>10} {'GAN better':>12}")
for metric in metric_order:
    item = comparison["metric_comparison"][metric]
    print(f"{metric:<8} {item['unet']:10.6f} {item['gan']:10.6f} {item['gan_minus_unet']:10.6f} {str(item['gan_better']):>12}")


本次结果：

- GAN 在 L1、MSE、PSNR、SSIM 上有小幅提升。
- GAN 在 Pearson 上低于 U-Net。
- 这个结果说明 adversarial refinement 有一定图像质量收益，同时也可能带来相关性保真的损失。


## 9. 结果图

下面展示已经保存的 comparison figures。

### Metrics

![comparison metrics](../outputs/gpu_phase_comparison/comparison/comparison_metrics.png)

### Reconstruction Samples

![comparison samples](../outputs/gpu_phase_comparison/comparison/comparison_samples.png)


In [ ]:
for relative in [
    "comparison/comparison_metrics.png",
    "comparison/comparison_samples.png",
]:
    path = run_dir / relative
    image = Image.open(path)
    print(relative, image.size)
    assert image.size[0] > 0 and image.size[1] > 0


## 10. 总结

当前项目已经实现并跑通了一套 coherent scattering reconstruction prototype：先用相位屏或粒子遮挡生成污染光场，再经过自由空间传播和 D2NN intensity readout 形成神经网络输入，随后用 U-Net 进行监督重建，并在 U-Net checkpoint 基础上加入 PatchGAN 做对抗细化。GPU 实验显示 U-Net 已经能够恢复数字结构，GAN 对部分指标有小幅提升，同时 Pearson 有下降。
